### Links and Resources
| Concept / Command | Description |
|-------------------|-------------|
| [Schema Enforcement](https://docs.databricks.com/en/delta/schema-validation.html) | Delta rejects writes whose schema doesn't match the table by default |
| [Schema Evolution](https://docs.databricks.com/en/delta/update-schema.html) | How to safely add, rename, and drop columns over time |
| [mergeSchema](https://docs.databricks.com/en/delta/update-schema.html#add-columns) | Write option that adds new columns from the source DataFrame |
| [overwriteSchema](https://docs.databricks.com/en/delta/update-schema.html#replace-table-schema) | Write option that completely replaces the table schema |
| [ALTER TABLE … ADD COLUMN](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-alter-table.html) | DDL to add one or more columns without rewriting data |
| [ALTER TABLE … DROP COLUMN](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-alter-table.html) | DDL to drop a column — requires column mapping enabled |
| [ALTER TABLE … RENAME COLUMN](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-alter-table.html) | DDL to rename a column — requires column mapping enabled |
| [Column Mapping](https://docs.databricks.com/en/delta/column-mapping.html) | Feature that decouples logical column names from physical Parquet field names |
| [Type Widening](https://docs.databricks.com/en/delta/type-widening.html) | Automatic safe type promotions: `int → long`, `float → double`, etc. |
| [autoMerge (MERGE)](https://docs.databricks.com/en/delta/update-schema.html#enable-schema-evolution-in-merge) | Session flag that enables schema evolution inside MERGE operations |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Schema Evolution

Real-world tables grow over time: new business requirements add columns, old ones are renamed or retired, data types need to widen as volumes increase.  
Delta Lake manages this lifecycle through two complementary mechanisms:

| Mechanism | What it does |
|-----------|--------------|
| **Schema enforcement** | Rejects writes whose schema doesn't match the table — the safe default |
| **Schema evolution** | Controlled ways to change the schema without breaking existing data |

### How schema changes land in the log

Every schema change writes a new `metaData` action to the `_delta_log/`.  
Older Parquet files keep their original physical layout — Delta maps logical column names to physical fields transparently.  
For columns added after the initial write, older files simply return `null` for those columns at query time.

```
v0 files: [product_id, name, price]          ← no discount column
v3 files: [product_id, name, price, discount] ← new column present

Query at v3: SELECT product_id, name, price, discount FROM table
  → v0 files: discount = null   (column absent from old file)
  → v3 files: discount = 0.10   (column present in new file)
```

## Setup — Create the Demo Table

In [ ]:
from pyspark.sql import Row, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
from delta.tables import DeltaTable

TABLE = "population_metrics.default.product_inventory"

rows = [
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",           category="Electronics", price=39.99,  in_stock=True),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",         category="Furniture",   price=89.95,  in_stock=True),
]

spark.createDataFrame(rows).write.format("delta").mode("overwrite").saveAsTable(TABLE)

dt = DeltaTable.forName(spark, TABLE)
print(f"Table created — version 0.")
print("\nInitial schema:")
spark.table(TABLE).printSchema()
display(spark.table(TABLE).orderBy("product_id"))

# Part 1 — Schema Enforcement

Delta Lake validates every write against the table's registered schema.  
If the incoming DataFrame has columns not present in the table, or is missing required columns, the write is rejected with an `AnalysisException` — no data is written and the table is unchanged.

This is the **safe default**: it prevents silent data corruption from upstream schema drift.

In [ ]:
# DataFrame with an extra column 'discount' not present in the table
df_extra_col = spark.createDataFrame([
    Row(product_id=7, name="Webcam HD", category="Electronics", price=79.00, in_stock=True, discount=0.10)
])

print("Attempting append with an extra 'discount' column (no mergeSchema)...")
try:
    df_extra_col.write.format("delta").mode("append").saveAsTable(TABLE)
    print("Write succeeded (unexpected)")
except Exception as e:
    print(f"Write rejected — schema mismatch detected.")
    print(f"Error type : {type(e).__name__}")
    # Truncate the message for readability
    print(f"Error      : {str(e)[:300]}...")

In [ ]:
# DataFrame with a missing column
df_missing_col = spark.createDataFrame([
    Row(product_id=7, name="Webcam HD", category="Electronics", price=79.00)
    # in_stock is missing
])

print("Attempting append with a missing 'in_stock' column...")
try:
    df_missing_col.write.format("delta").mode("append").saveAsTable(TABLE)
    print("Write succeeded (unexpected)")
except Exception as e:
    print(f"Write rejected.")
    print(f"Error type : {type(e).__name__}")
    print(f"Error      : {str(e)[:300]}...")

# Table is unchanged — enforcement means atomicity: either all or nothing
print(f"\nTable row count is still: {spark.table(TABLE).count()}")

# Part 2 — `mergeSchema`: Adding New Columns on Write

Setting `.option("mergeSchema", "true")` on an append or overwrite tells Delta to **add any new columns** from the source DataFrame to the table schema, then proceed with the write.

Existing columns in the table that are absent from the source are filled with `null` in the new files.  
Existing rows already in the table return `null` for the new column because the old Parquet files do not contain it.

| Row source | New column value |
|------------|------------------|
| Written before `mergeSchema` | `null` (column absent from old Parquet files) |
| Written after `mergeSchema` | The actual value from the source DataFrame |

In [ ]:
# Source DataFrame introduces a new 'discount' column
df_with_discount = spark.createDataFrame([
    Row(product_id=7, name="Webcam HD",       category="Electronics", price=79.00, in_stock=True, discount=0.10),
    Row(product_id=8, name="Mechanical Keyboard", category="Electronics", price=129.00, in_stock=True, discount=0.05),
])

# mergeSchema=true → Delta adds 'discount' to the schema and writes the rows
df_with_discount.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(TABLE)

print("Schema after mergeSchema append:")
spark.table(TABLE).printSchema()
# 'discount' is now part of the schema

print("\nTable data — old rows show null for 'discount':")
display(spark.table(TABLE).orderBy("product_id"))

### `mergeSchema` Rules

| Source has ... | Result |
|----------------|--------|
| Extra columns not in target | Added to the table schema |
| Missing columns that are in target | New files get `null` for those columns |
| A column with a **different type** | Write is still rejected — `mergeSchema` does not allow type changes |
| A column with a **compatible wider type** | Accepted only if type widening is enabled (see Part 7) |

> `mergeSchema` can also be enabled session-wide:  
> `spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")`

In [ ]:
# Confirm the schema change is recorded in the transaction log
dt = DeltaTable.forName(spark, TABLE)
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)
# version 1 is the WRITE with mergeSchema — operationParameters includes the updated schema

# Part 3 — `overwriteSchema`: Replace the Entire Schema

`overwriteSchema=true` combined with `mode("overwrite")` replaces both the data **and** the schema in one operation.  
Use this when the new DataFrame's schema is intentionally different from the existing table — not just an addition.

> **Warning:** This deletes all existing rows. The old data is gone (until VACUUM removes the files, time travel still works).

In [ ]:
# A completely different schema — removed 'in_stock', added 'sku' and 'rating'
df_new_schema = spark.createDataFrame([
    Row(product_id=1, sku="EL-0001", name="Laptop Pro 15", category="Electronics", price=1299.99, rating=4.8),
    Row(product_id=2, sku="EL-0002", name="Wireless Mouse",category="Electronics", price=25.49,  rating=4.5),
])

# Without overwriteSchema this would fail — the schemas don't match
df_new_schema.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print("Schema after overwriteSchema:")
spark.table(TABLE).printSchema()
display(spark.table(TABLE))

In [ ]:
# Recreate the original table for the remaining sections
spark.createDataFrame(rows).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)
print("Table restored to original schema.")
spark.table(TABLE).printSchema()

# Part 4 — `ALTER TABLE ADD COLUMN`

`ALTER TABLE … ADD COLUMN` is the SQL DDL way to extend the schema.  
It is a **metadata-only operation** — no Parquet files are rewritten.  
Delta writes a new `metaData` action to the `_delta_log/` with the updated schema.  
All existing rows return `null` for the new column.

You can add a column with:
- A data type
- `AFTER existing_column` to control position
- `COMMENT` for documentation
- `NOT NULL` constraint (only if the table has no existing rows that would violate it)

In [ ]:
# Add a single column — metadata-only, no file rewrite
spark.sql(f"""
    ALTER TABLE {TABLE}
    ADD COLUMN discount DOUBLE COMMENT 'Fractional discount applied at checkout (0.0–1.0)'
""")

print("Schema after ADD COLUMN discount:")
spark.table(TABLE).printSchema()
display(spark.table(TABLE).orderBy("product_id"))
# All existing rows have discount = null

In [ ]:
# Add multiple columns in one DDL statement
spark.sql(f"""
    ALTER TABLE {TABLE}
    ADD COLUMNS (
        sku          STRING  COMMENT 'Stock-keeping unit — unique product code',
        last_updated TIMESTAMP COMMENT 'Timestamp of the last row modification'
    )
""")

print("Schema after adding sku and last_updated:")
spark.table(TABLE).printSchema()

In [ ]:
# Populate the new columns via UPDATE
dt = DeltaTable.forName(spark, TABLE)

dt.update(set={
    "discount":     F.lit(0.0),
    "sku":          F.concat(F.lit("PRD-"), F.lpad(F.col("product_id").cast("string"), 4, "0")),
    "last_updated": F.current_timestamp(),
})

print("After populating the new columns:")
display(spark.table(TABLE).orderBy("product_id"))

### ADD COLUMN with Nested Position (`AFTER`)

By default new columns are appended to the end of the schema.  
Use `AFTER existing_column` to place a new column immediately after a specific existing column.

In [ ]:
# Insert 'brand' immediately after 'name'
spark.sql(f"""
    ALTER TABLE {TABLE}
    ADD COLUMN brand STRING COMMENT 'Manufacturer or brand name'
    AFTER name
""")

print("Schema with 'brand' positioned after 'name':")
spark.table(TABLE).printSchema()
# brand will appear between name and category

# Part 5 — Column Mapping

By default, Delta stores column names **physically** inside each Parquet file.  
This means renaming or dropping a column requires rewriting every Parquet file — expensive and disruptive.

**Column mapping** decouples the **logical name** (what SQL sees) from the **physical field name** (what is stored in Parquet).  
Delta assigns each column a stable internal ID; renaming a column only updates the `metaData` entry in the log — no file rewrite.

```
Without column mapping:
  Parquet field: "name"  →  SQL column: "name"
  RENAME name → product_name  requires rewriting all Parquet files

With column mapping (mode = 'name'):
  Parquet field: "name"  →  logical SQL column: "product_name"
  RENAME only updates _delta_log/ metaData — zero file I/O
```

Column mapping is required for:
- `ALTER TABLE … RENAME COLUMN`
- `ALTER TABLE … DROP COLUMN`

On Databricks, column mapping mode `'name'` is the standard and is enabled by upgrading the table protocol.

In [ ]:
# Enable column mapping on the existing table
# This upgrades the Delta protocol — a one-way operation
spark.sql(f"""
    ALTER TABLE {TABLE}
    SET TBLPROPERTIES (
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5',
        'delta.columnMapping.mode' = 'name'
    )
""")

# Confirm the feature is now active
dt = DeltaTable.forName(spark, TABLE)
detail = dt.detail().collect()[0]
props  = detail["properties"] or {}

print("Column mapping properties:")
for k in ["delta.columnMapping.mode", "delta.minReaderVersion", "delta.minWriterVersion"]:
    print(f"  {k:40s}: {props.get(k, detail.get(k, 'N/A'))}")

print("\nTable features:")
print(f"  {detail['tableFeatures']}")

# Part 6 — `ALTER TABLE RENAME COLUMN`

With column mapping enabled, renaming is a pure metadata operation — the physical Parquet field name stays unchanged; only the logical name in `_delta_log/` is updated.

All queries, views, and downstream jobs that use the **new** name will work immediately.  
Queries using the **old** name will fail — this is intentional (schema enforcement protects against stale references).

In [ ]:
print("Schema before RENAME:")
spark.table(TABLE).printSchema()

In [ ]:
# Rename 'name' → 'product_name'  (more descriptive)
spark.sql(f"ALTER TABLE {TABLE} RENAME COLUMN name TO product_name")

# Rename 'in_stock' → 'is_available'  (align with new naming convention)
spark.sql(f"ALTER TABLE {TABLE} RENAME COLUMN in_stock TO is_available")

print("Schema after renaming 'name' → 'product_name' and 'in_stock' → 'is_available':")
spark.table(TABLE).printSchema()

display(spark.table(TABLE).orderBy("product_id"))

In [ ]:
# Old name is now invalid
try:
    spark.sql(f"SELECT name FROM {TABLE} LIMIT 1").show()
except Exception as e:
    print(f"Old column name 'name' rejected: {type(e).__name__}")
    print(f"  {str(e)[:200]}")

# New name works
print("\nNew column name 'product_name' works fine:")
display(spark.sql(f"SELECT product_id, product_name, is_available FROM {TABLE} ORDER BY product_id"))

# Part 7 — `ALTER TABLE DROP COLUMN`

Dropping a column also requires column mapping.  
Like RENAME, it is a metadata-only operation — the Parquet files are not rewritten.  
The physical field still exists in old files; Delta simply stops projecting it into query results.

> **Caution:** Dropped columns are immediately invisible to all queries. If any view, job, or downstream consumer references the column by name it will break.

In [ ]:
print("Columns before DROP:")
print(spark.table(TABLE).columns)

In [ ]:
# Drop the 'brand' column (it was added earlier but never populated)
spark.sql(f"ALTER TABLE {TABLE} DROP COLUMN brand")

# Drop multiple columns at once
spark.sql(f"ALTER TABLE {TABLE} DROP COLUMNS (sku, last_updated)")

print("Columns after DROP:")
print(spark.table(TABLE).columns)

print("\nSchema after dropping brand, sku, last_updated:")
spark.table(TABLE).printSchema()

### Inspecting Schema Changes in History

Every `ALTER TABLE` statement that changes the schema adds a `CHANGE COLUMN` or `ADD COLUMNS` or `REPLACE COLUMNS` operation to the transaction log.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)

# Part 8 — `ALTER TABLE CHANGE COLUMN`: Type Evolution and Comments

`CHANGE COLUMN` modifies an existing column's metadata: its data type, nullability, comment, or position.  

### Type widening — safe promotions

Delta allows **widening** type changes — promotions that can represent all existing values without data loss:

| From | Allowed widening targets |
|------|-------------------------|
| `BYTE` | `SHORT`, `INT`, `LONG`, `FLOAT`, `DOUBLE`, `DECIMAL` |
| `SHORT` | `INT`, `LONG`, `FLOAT`, `DOUBLE`, `DECIMAL` |
| `INT` | `LONG`, `FLOAT`, `DOUBLE`, `DECIMAL` |
| `LONG` | `FLOAT`, `DOUBLE`, `DECIMAL` |
| `FLOAT` | `DOUBLE` |
| `DATE` | `TIMESTAMP` |

Narrowing casts (`LONG → INT`, `DOUBLE → FLOAT`) are never allowed — they could silently truncate values.

In [ ]:
# Add a 'stock_count' column as INT, then widen to LONG to handle large warehouses
spark.sql(f"ALTER TABLE {TABLE} ADD COLUMN stock_count INT COMMENT 'Units available in warehouse'")
print("Added stock_count as INT.")

# Widen INT → LONG — metadata-only, no file rewrite needed
spark.sql(f"""
    ALTER TABLE {TABLE}
    CHANGE COLUMN stock_count stock_count LONG COMMENT 'Units available in warehouse (widened to LONG)'
""")
print("Widened stock_count from INT to LONG.")

print("\nRelevant part of updated schema:")
schema = spark.table(TABLE).schema
for f in schema.fields:
    if f.name == "stock_count":
        print(f"  {f.name}: {f.dataType} — nullable={f.nullable}")

In [ ]:
# Narrowing is rejected
print("Attempting LONG → INT (narrowing — should be rejected):")
try:
    spark.sql(f"""
        ALTER TABLE {TABLE}
        CHANGE COLUMN stock_count stock_count INT
    """)
    print("Unexpectedly succeeded")
except Exception as e:
    print(f"Rejected: {type(e).__name__}")
    print(f"  {str(e)[:200]}")

In [ ]:
# Update the COMMENT on an existing column — CHANGE COLUMN syntax
spark.sql(f"""
    ALTER TABLE {TABLE}
    CHANGE COLUMN price price DOUBLE COMMENT 'List price in USD — excludes applicable discounts'
""")

# View column-level comments via DESCRIBE EXTENDED
display(
    spark.sql(f"DESCRIBE EXTENDED {TABLE}")
    .filter(F.col("col_name").isin("price", "stock_count", "discount"))
)

# Part 9 — Schema Evolution in MERGE

By default, MERGE enforces the target table's schema — source columns not present in the target cause the operation to fail.  
Enable schema evolution for MERGE by setting the session flag:

```python
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
```

With `autoMerge` enabled:
- New columns in the source are **added** to the target schema.
- Existing target columns absent from the source remain unchanged in matched rows.

In [ ]:
# Without autoMerge — source has a column the target doesn't know about
source_with_new_col = spark.createDataFrame([
    Row(product_id=1, product_name="Laptop Pro 15", category="Electronics",
        price=1099.99, is_available=True, discount=0.05, supplier="TechCorp"),  # 'supplier' is new
])

dt = DeltaTable.forName(spark, TABLE)

print("MERGE without autoMerge (source has extra 'supplier' column):")
try:
    (
        dt.merge(source_with_new_col, "target.product_id = source.product_id")
        .whenMatchedUpdateAll()
        .execute()
    )
    print("Succeeded (unexpected)")
except Exception as e:
    print(f"Rejected: {type(e).__name__}")
    print(f"  {str(e)[:250]}")

In [ ]:
# Enable autoMerge for the session
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

dt = DeltaTable.forName(spark, TABLE)

(
    dt.merge(source_with_new_col, "target.product_id = source.product_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Schema after MERGE with autoMerge — 'supplier' column added:")
spark.table(TABLE).printSchema()

display(
    spark.table(TABLE)
    .select("product_id", "product_name", "price", "supplier")
    .orderBy("product_id")
)
# product_id 1 has supplier = TechCorp, all others have supplier = null

In [ ]:
# Reset autoMerge to default (off) after this section
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

# Part 10 — Nested Columns (Struct Evolution)

Delta schema evolution works on **nested structs** too.  
You can add fields inside a struct column using dot notation in `ALTER TABLE ADD COLUMN`.

This is useful for grouping related metadata fields without flattening them into top-level columns.

In [ ]:
# Add a top-level struct column 'shipping' with nested fields
spark.sql(f"""
    ALTER TABLE {TABLE}
    ADD COLUMN shipping STRUCT<
        weight_kg : DOUBLE  COMMENT 'Product weight in kilograms',
        fragile   : BOOLEAN COMMENT 'True if special packaging is required'
    > COMMENT 'Shipping attributes'
""")

print("Schema after adding 'shipping' struct column:")
spark.table(TABLE).printSchema()

In [ ]:
# Add a new field INSIDE the struct — dot notation
spark.sql(f"""
    ALTER TABLE {TABLE}
    ADD COLUMN shipping.requires_signature BOOLEAN
    COMMENT 'True if recipient must sign on delivery'
""")

print("Schema after adding field inside 'shipping' struct:")
spark.table(TABLE).printSchema()

In [ ]:
# Populate the struct using UPDATE with a struct literal
dt = DeltaTable.forName(spark, TABLE)

dt.update(
    condition="product_id = 1",
    set={"shipping": F.struct(
        F.lit(2.5).alias("weight_kg"),
        F.lit(False).alias("fragile"),
        F.lit(True).alias("requires_signature")
    )}
)

print("Shipping struct for product_id = 1:")
display(
    spark.table(TABLE)
    .filter("product_id = 1")
    .select("product_id", "product_name", "shipping",
            "shipping.weight_kg", "shipping.fragile", "shipping.requires_signature")
)

# Part 11 — Schema at Any Historical Version

Because each schema change writes a `metaData` action to the `_delta_log/`, time travel also travels back through schemas.  
Reading an older version returns only the columns that existed at that version.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)
max_version = dt.history(1).collect()[0]["version"]

print("Schema column count at each version:\n")
location = dt.detail().collect()[0]["location"]

for v in range(max_version + 1):
    df_v = spark.read.format("delta").option("versionAsOf", v).load(location)
    op   = dt.history().filter(f"version = {v}").collect()[0]["operation"]
    cols = df_v.columns
    print(f"  v{v:2d}  {op:45s}  columns={len(cols)}: {cols}")

# Summary

## Schema Change Methods

| Method | How | File rewrite? | Use case |
|--------|-----|:-------------:|----------|
| `mergeSchema=true` (write option) | Add columns from source DataFrame on append/overwrite | No | Incremental ingestion with evolving upstream schemas |
| `overwriteSchema=true` (write option) | Replace schema and data | Yes (full overwrite) | Complete schema redesign |
| `ALTER TABLE ADD COLUMN` | DDL — add one or more columns | No | Deliberate, reviewed schema extension |
| `ALTER TABLE RENAME COLUMN` | DDL — rename a column | No (**requires column mapping**) | Naming convention changes |
| `ALTER TABLE DROP COLUMN` | DDL — remove a column | No (**requires column mapping**) | Retiring unused fields |
| `ALTER TABLE CHANGE COLUMN` | DDL — widen type, change comment/nullability | No | Type promotions, documentation |
| MERGE with `autoMerge` | Session flag — add source columns to target during MERGE | No | CDC ingestion with expanding source schemas |

## Column Mapping

```sql
-- Enable on an existing table (one-way upgrade)
ALTER TABLE table_name SET TBLPROPERTIES (
    'delta.minReaderVersion'     = '2',
    'delta.minWriterVersion'     = '5',
    'delta.columnMapping.mode'   = 'name'
);
```
- Required for `RENAME COLUMN` and `DROP COLUMN`.
- Both operations become zero-I/O metadata updates.

## Safe Type Widening

```sql
ALTER TABLE t CHANGE COLUMN qty qty LONG;   -- INT → LONG  ✓
ALTER TABLE t CHANGE COLUMN qty qty INT;    -- LONG → INT  ✗ rejected
```

## `autoMerge` for MERGE

```python
# Session-level — applies to all MERGE operations in this SparkSession
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# Reset when done
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")
```

## What to Expect for Existing Rows After a Schema Change

| Change | Old rows return |
|--------|----------------|
| Column added | `null` for the new column |
| Column renamed | New name works; old name rejected |
| Column dropped | Column no longer visible; still in old Parquet files |
| Type widened | Old values read and automatically cast to the wider type |